# 验证 · Latent-SFT code 训练流程(小规模 6 步 + loss 收敛)
**A100 → Run all**。小规模(200 行 · S1=2/S2=3ep)验证 6 步管线跑通、stage2 loss 下降。真训练用 `handoff/train_lsft_code.sh`。
> 6 步串行, 含'老代码跑新 Colab'兼容补丁(scatter dtype / force-llama / tokenizer / config)。

In [ ]:
# 验证走 handoff 脚本 VERIFY 模式(6 步太长, 复用已推 mirror 的 .sh; 之后画 loss)
import os, subprocess
if not os.path.exists("/content/lrm/.git"): subprocess.run("git clone -q %s /content/lrm" % "https://github.com/ruijiezh67/LRM_colab_tasks.git", shell=True)
os.environ["WORK"]="/content/crux_retrain_work"; os.environ["VERIFY"]="1"
print(">>> 跑 train_lsft_code.sh VERIFY=1(自包含: floor+6步小规模)。日志 tee /content/train.log ...")
!cd /content && VERIFY=1 WORK=/content/crux_retrain_work bash /content/lrm/handoff/train_lsft_code.sh 2>&1 | tee /content/train.log


In [ ]:
# stage2 loss 收敛画图

import re, glob, matplotlib.pyplot as plt
def show_loss(logpath, title):
    txt = open(logpath, encoding="utf-8", errors="ignore").read()
    ls = [float(x) for x in re.findall(r"(?:'loss'|train_loss|loss)[=:'\s]+([0-9]+\.[0-9]+)", txt)]
    ls = [x for x in ls if x < 1e4]
    if len(ls) >= 2:
        plt.figure(figsize=(6,3)); plt.plot(ls, marker="."); plt.title(title+" · loss"); plt.xlabel("log step"); plt.ylabel("loss"); plt.grid(alpha=.3); plt.show()
        print(f"loss: {ls[0]:.3f} -> {ls[-1]:.3f} ({len(ls)} 点)")
        print("✅ [PASS] 训练流程正确: loss 下降(收敛趋势)" if ls[-1] < ls[0] else "⚠ [WARN] 跑通但 loss 没降 → 查 lr/数据/配方")
    else:
        print("⚠ 没抓到 loss 序列 → 看上面日志确认训练是否真启动")

import glob
logs=glob.glob("/content/crux_retrain_work/lsft_run_distill_stage2*.log")
show_loss(logs[-1] if logs else "/content/train.log", "LSFT-code stage2")